Creating Snowpark Session and Initializing warehouse, Database and Schema

In [ ]:
from snowflake.snowpark import Session
session = Session.builder.getOrCreate()
session


session.sql("use database airbnb_db").collect()
session.sql("use schema dbsc_staging").collect()

session.sql("select current_warehouse(), current_database(), current_schema()")

In [ ]:
bookings_df = session.table("stg_airbnb_bookings")
hosts_df = session.table("stg_airbnb_hosts")
listings_df = session.table("stg_airbnb_listings")

bookings_df.show()
hosts_df.show()
listings_df.show()

Converting Snowpark Dataframe into Pnadas Dataframe, After doing this, the lazy evaluation feature of snowpark will completely end.
Use .to_pandas() directly with dataframe to convert into pandas dataframe

bookings_pd = bookings_df.to_pandas()

In [ ]:
# Group bookings_df by BOOKING_STATUS, 
# and compute total revenue as the sum of BOOKING_AMOUNT + CLEANING_FEE + SERVICE_FEE, 
# aliased as TOTAL_REVENUE. Order by TOTAL_REVENUE descending.

from snowflake.snowpark.functions import * 

total_revenue_by_status_df = bookings_df.groupBy(col("BOOKING_STATUS")).agg(
sum(col("BOOKING_AMOUNT") + col("CLEANING_FEE") + col("SERVICE_FEE")).alias("TOTAL_REVENUE")
).orderBy(col("TOTAL_REVENUE").desc())

total_revenue_by_status_df.show()

In [ ]:
top_listings_df = bookings_df.groupBy(
col("LISTING_ID")
).agg(
count(col("BOOKING_ID")).alias("TOTAL_BOOKINGS")
).orderBy(col("TOTAL_BOOKINGS").desc()).limit(10)

top_listings_df.show()

In [ ]:
monthly_booking_df = bookings_df.groupBy(
    year(col("BOOKING_DATE")).alias("BOOKING_YEAR"),
    month(col("BOOKING_DATE")).alias("BOOKING_MONTH")
).agg(
    count(col("BOOKING_ID")).alias("TOTAL_BOOKINGS")
).orderBy(
    col("BOOKING_YEAR"),
    col("BOOKING_MONTH")
)

monthly_booking_df.show(12)

In [ ]:
#sql version
from snowflake.snowpark.window import Window

listings_df.create_or_replace_temp_view("listing_temp")
above_avg_listing = session.sql(
"""
select listing_id
from listing_temp
where price_per_night > (select avg(price_per_night) from listing_temp)
order by listing_id
"""
)
above_avg_listing.show()



In [ ]:
# #snowpark version
# # approach-1
window_spec = Window.partitionBy()

above_avg_listing = listings_df.withColumn(
    "AVG_PRICE", avg(col("PRICE_PER_NIGHT")).over(window_spec)
).filter(
    col("PRICE_PER_NIGHT") > col("AVG_PRICE")
).select(
    col("LISTING_ID")
).orderBy(
    col("LISTING_ID").asc()
)


# above_avg_listing.show()

#approach_2
avg_price_df = listings_df.agg(avg(col("price_per_night")).alias("avg_price"))
# avg_price_df.show()

listings_joined_avg_price_df = (
    listings_df.cross_join(avg_price_df)
    .filter(col("PRICE_PER_NIGHT") > col("AVG_PRICE"))
    .select(col("LISTING_ID"))
    .orderBy(col("LISTING_ID"))
)

listings_joined_avg_price_df.show()


In [ ]:
# First, join bookings_df and listings_df on LISTING_ID
# Then join that result with hosts_df on HOST_ID
# Group by something that identifies the host (think: HOST_ID, or HOST_NAME if you want it readable)
# Aggregate total revenue — reuse your revenue formula from Exercise 1 (BOOKING_AMOUNT + CLEANING_FEE + SERVICE_FEE)

bookings_listings_hosts_df = (
    bookings_df.join(
        listings_df, 
        bookings_df["LISTING_ID"] == listings_df["LISTING_ID"],
        how="inner"
    ).join(
        hosts_df, 
        listings_df["HOST_ID"] == hosts_df["HOST_ID"],
        how="inner"
    )
)

result_df = (
    bookings_listings_hosts_df.groupBy(
        hosts_df["HOST_ID"].alias("HOST_ID"), 
        col("HOST_FIRST_NAME").alias("FIRST_NAME"),
        col("HOST_LAST_NAME").alias("LAST_NAME")
    )
    .agg(
        sum(
            col("BOOKING_AMOUNT") + col("CLEANING_FEE") + col("SERVICE_FEE")
        ).alias("TOTAL_REVENUE")
    )
    .orderBy(
        # col("HOST_ID"), 
        # col("FIRST_NAME"),
        # col("LAST_NAME")
        col("TOTAL_REVENUE").desc()
    )
)
result_df.show()



In [ ]:
#Which CITY generates the most revenue? (single join: bookings_df → listings_df)

most_revenued_city_df = (
    bookings_df.join(
        listings_df,
        bookings_df["LISTING_ID"] == listings_df["LISTING_ID"],
        how = "inner"
    )
    .group_by(col("CITY"))
    .agg(
        sum(
            col("BOOKING_AMOUNT") + col("CLEANING_FEE") + col("SERVICE_FEE")
        ).alias("TOTAL_REVENUE")
    )
    .orderBy(
        col("TOTAL_REVENUE").desc()
    )
    .limit(1)
)
most_revenued_city_df.show()


In [ ]:
result_df = (
    #join bookings and listings df
    bookings_df.join(
        listings_df,
        bookings_df["LISTING_ID"] == listings_df["LISTING_ID"],
        how="inner"
    )
    .groupBy(col("ROOM_TYPE"))
    .agg(
        round(avg(col("NIGHTS_BOOKED")), 2).alias("AVG_NIGHTS_BOOKED")
    )
    .orderBy(
        col("AVG_NIGHTS_BOOKED").desc()
    )
)
result_df.show()

In [ ]:
#Find superhosts (IS_SUPERHOST = TRUE) 
#and compare their average revenue per listing against non-superhosts.

from snowflake.snowpark.window import Window

#first join hosts and listings

full_df = (
    bookings_df.join(
        listings_df,
        bookings_df["LISTING_ID"] == listings_df["LISTING_ID"],
        how = "inner"
    ).join(
        hosts_df,
        listings_df["HOST_ID"] == hosts_df["HOST_ID"],
        how = "inner"
    )  
)

#full_df.show()


listing_revenue_df = (
    full_df.groupBy(
        listings_df["LISTING_ID"],
        col("IS_SUPERHOST")
    ).agg(
       sum((col("BOOKING_AMOUNT") + col("CLEANING_FEE") + col("SERVICE_FEE"))).alias("TOTAL_LISTING_REVENUE") 
    )
)

#listing_revenue_df.show()

comparison_df = (
    listing_revenue_df.groupBy(
        col("IS_SUPERHOST")
    ).agg(
        round(avg(col("TOTAL_LISTING_REVENUE")), 2).alias("AVG_REVENUE_PER_LISTING")
    )
    .orderBy(col("AVG_REVENUE_PER_LISTING"))
)
comparison_df.show()




In [ ]:
from snowflake.snowpark.window import Window

#listing revenue per city
full_df = (
    bookings_df.join(
        listings_df,
        bookings_df["LISTING_ID"] == listings_df["LISTING_ID"],
        how = "inner"
    ).join(
        hosts_df,
        listings_df["HOST_ID"] == hosts_df["HOST_ID"],
        how = "inner"
    )
)
#listing_host_df.show()

listing_revenue_per_city_df = (
    full_df.groupBy(
        listings_df["LISTING_ID"].alias("LISTING_ID"),
        col("CITY")
    ).agg(
        sum(
            (col("BOOKING_AMOUNT") + col("CLEANING_FEE") + col("SERVICE_FEE"))
        ).alias("TOTAL_REVENUE")
    )
)
#listing_revenue_per_city_df.show()
window_spec = Window.partitionBy(
    col("CITY")
).orderBy(col("TOTAL_REVENUE").desc())
ranked_df = (

    listing_revenue_per_city_df.withColumn(
        "LISTING_RANK_PER_CITY", 
        rank().over(window_spec)
    )
)
#ranked_df.show(10)

# #top 3 revenue listing by city 

ranked_df.filter(
    col("LISTING_RANK_PER_CITY") <= 3 
).orderBy(
    col("CITY"),
    col("LISTING_RANK_PER_CITY")
).show(30)

